In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install gensim pandas numpy -q

import os
import ast
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
import warnings
warnings.filterwarnings("ignore")

PREPROCESSED_PATH     = "/content/drive/MyDrive/dataset_preprocessed.csv"
MODEL_DIR             = "/content/drive/MyDrive/fake_news_models"
WORD2VEC_PATH         = os.path.join(MODEL_DIR, "word2vec_model.model")
EMBEDDING_MATRIX_PATH = os.path.join(MODEL_DIR, "embedding_matrix.npy")
EMBEDDING_LABELS_PATH = os.path.join(MODEL_DIR, "embedding_labels.npy")

os.makedirs(MODEL_DIR, exist_ok=True)
print("Setup complete!")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 70.6 MB/s eta 0:00:00
Setup complete!


In [2]:
print("Loading preprocessed dataset")
df = pd.read_csv(PREPROCESSED_PATH)

Loading preprocessed dataset


In [3]:
df.head()

,title,text,unreliable,author,word_count,char_length,preprocessingText2,cleanText
0,WARNING: A Pivotal Moment For The Stock Market...,WARNING: A Pivotal Moment For The Stock Market...,1,Anonymous Coward (UID 72071746),43,228,"['warn', 'pivotal', 'moment', 'stock', 'market...",warn pivotal moment stock market may page mail...
1,"Trump, top defense officials, discuss North Ko...",WASHINGTON - U.S. President Donald Trump met ...,0,Unknown,85,426,"['washington', 'u', 'president', 'donald', 'tr...",washington u president donald trump meet tuesd...
2,British civil servants' union calls nationwide...,LONDON - British civil servants will vote nex...,0,Unknown,304,1491,"['london', 'british', 'civil', 'servant', 'vot...",london british civil servant vote next month n...
3,A**hole Of The Day – Michele Bachmann: Muslim...,Michele Bachmann has been pretty quiet since l...,1,Unknown,633,2980,"['michele', 'bachmann', 'pretty', 'quiet', 'si...",michele bachmann pretty quiet since leave offi...
4,’Deport Fat People’ Posters Appear At CU Bould...,Posters calling on Donald Trump to “Deport Fat...,0,Lucas Nolan,123,578,"['poster', 'call', 'donald', 'trump', 'deport'...",poster call donald trump deport fat people app...


In [4]:
def parse_tokens(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except Exception:
            return val.split()
    return []


In [5]:
df["preprocessingText2"] = df["preprocessingText2"].apply(parse_tokens)
corpus = []
for tokens in df["preprocessingText2"]:
    if isinstance(tokens, list) and len(tokens) > 0:
        corpus.append(tokens)

In [6]:
print(f"Total documents in corpus: {len(corpus)}")
print(f"Sample (first doc, first 10 tokens): {corpus[0][:10]}")

Total documents in corpus: 58417
Sample (first doc, first 10 tokens): ['warn', 'pivotal', 'moment', 'stock', 'market', 'may', 'page', 'mail', 'question', 'comment']


In [7]:
import os

print(os.listdir("/content/drive/MyDrive/fake_news_models"))

[]


In [8]:
if os.path.exists(WORD2VEC_PATH):
    print("Loading existing Word2Vec model")
    w2v_model = Word2Vec.load(WORD2VEC_PATH)


else:
  print("Training Word2Vec")
  w2v_model = Word2Vec(
      sentences=corpus,
      vector_size=100,
      window=5,
      min_count=2,
      workers=4,
      epochs=10,
      sg=1,
      seed=42
    )
  print("Word2Vec training complete.")
  w2v_model.save(WORD2VEC_PATH)
  print("Saved Word2Vec model to Drive!")

print(f"Vocabulary size: {len(w2v_model.wv):,} unique words")
print(f"Vector size per word: {w2v_model.vector_size}")

Training Word2Vec
Word2Vec training complete.
Saved Word2Vec model to Drive!
Vocabulary size: 120,151 unique words
Vector size per word: 100


In [9]:
#inspecting the most similar words for selected terms to verify that the Word2Vec model has learned meaningful semantic relationships
query_words = ["fake", "real", "breaking", "trump", "government"]

for word in query_words:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        print(f"\nMost similar to '{word}':")
        for similar_word, score in similar:
            print(f"  {similar_word:<20} similarity: {score:.4f}")
    else:
        print(f"\n'{word}' not in vocabulary (was removed during preprocessing).")


Most similar to 'fake':
  propagator           similarity: 0.7697
  newsthere            similarity: 0.7503
  httpstcotssgvnyuct   similarity: 0.7396
  agendatrump          similarity: 0.7378
  mediahe              similarity: 0.7312

Most similar to 'real':
  estate               similarity: 0.7440
  mogulreality         similarity: 0.7272
  mogulturnedpolitician similarity: 0.7179
  tenx                 similarity: 0.7015
  unshackle            similarity: 0.6873

Most similar to 'breaking':
  break                similarity: 0.6898
  broke                similarity: 0.5569
  truthfeednews        similarity: 0.5528
  usapoliticsnow       similarity: 0.5454
  shoesthe             similarity: 0.5426

Most similar to 'trump':
  donald               similarity: 0.8681
  trumps               similarity: 0.8600
  itthroughout         similarity: 0.7552
  itsadly              similarity: 0.7482
  trumppresident       similarity: 0.7474

Most similar to 'government':
  hackersthe           

In [10]:
def article_to_vector(tokens, model, vector_size=100):
    known_tokens = [word for word in tokens if word in model.wv]

    if len(known_tokens) == 0:
        return np.zeros(vector_size)

    vectors = np.array([model.wv[word] for word in known_tokens])
    return vectors.mean(axis=0)


In [11]:
if os.path.exists(EMBEDDING_MATRIX_PATH) and os.path.exists(EMBEDDING_LABELS_PATH):
    print("Loading existing embedding matrix and labels")
    embedding_matrix = np.load(EMBEDDING_MATRIX_PATH)
    labels = np.load(EMBEDDING_LABELS_PATH, allow_pickle=True)
else:
    print("Building embedding matrix")
    embedding_matrix = np.array([
        article_to_vector(tokens, w2v_model, vector_size=100)
        for tokens in df["preprocessingText2"]
    ])
    labels = df["unreliable"].values.astype(np.float32)
    np.save(EMBEDDING_MATRIX_PATH, embedding_matrix)
    np.save(EMBEDDING_LABELS_PATH, labels)
    print("Saved embedding matrix and labels to Drive")

print("Embedding matrix shape:", embedding_matrix.shape)
print("Labels shape:", labels.shape)
print("\nWord2Vec and embeddings saved to Drive")

Building embedding matrix
Saved embedding matrix and labels to Drive
Embedding matrix shape: (58417, 100)
Labels shape: (58417,)

Word2Vec and embeddings saved to Drive
